# IBKR API notebook

#### Connection

In [ ]:
from ib_async import *
import pandas as pd
import logging
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.DEBUG)

## Request Historical data

#### Choose your contract

In [ ]:
contract = Stock('AAPL', 'SMART', 'USD')

In [ ]:
contract = Index('NDX', 'NASDAQ', 'USD')
print("Contract details:")
print("Symbol:", contract.symbol)
print("Exchange:", contract.exchange)
print("Currency:", contract.currency)
print("Trading Class:", contract.tradingClass)  
print("Local Symbol:", contract.localSymbol)
print("Contract ID:", contract.conId)

In [ ]:
contract = Forex('EURUSD')

#### Check first data timestamp available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=True)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
print(f"First date of data available: {formatted_time}")

#### Request historical data function

In [ ]:
save_path = "./database/AAPL_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)
# Retrieve the first date from your existing DataFrame
first_date = retrieved_df.iloc[0]['date']  # Assuming 'date' is the column name
end_date = first_date.strftime('%Y%m%d %H:%M:%S')  # Format as 'yyyyMMdd HH:mm:ss'
print(f"First date in the DataFrame: {first_date}")
print(f"End date: {end_date}")

In [ ]:
#today's date
end_date = pd.Timestamp.now().strftime('%Y%m%d %H:%M:%S')

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [ ]:
historical_data_interval = '10 secs' 
request_duration = '1 Y'
bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1,
        timeout = 0)

In [ ]:
bars[0]

Convert the list of bars to a data frame and print the first and last rows:

In [ ]:
df = util.df(bars)
print("DataFrame shape:", df.shape)

display(df.head(n=20))
display(df.tail(n=20))

Save your pulled data in a dataframe

Compression possibilities sorted by compression ratio from the lowest to the highest : 
- `snappy`

- `gzip`

- `brotli`

#### Checking if volume and average columns are empty or not and remove it if empty

In [ ]:
# Check if the 'volume' column is empty (all values are 0.0)
if (df['volume'] == 0.0).all():
    df = df.drop(columns=['volume'])  # Drop the 'volume' column
    print("The 'volume' column was empty and has been removed.")

# Check if the 'average' column is empty (all values are 0.0)
if (df['average'] == 0.0).all():
    df = df.drop(columns=['average'])  # Drop the 'average' column
    print("The 'average' column was empty and has been removed.")

# Display the updated DataFrame
display(df.head(n=30))

Construction du nom du fichier et sauvegarde en `.parquet` dans le dossier `database`

Structure du nom du fichier : `Symbol_Interval_StartDate_EndDate.parquet`

In [ ]:
# Récupérer la devise et l'unité de temps
symbol = contract.symbol

# Construire le nom du fichier
start_date = df['date'].iloc[0].strftime('%Y%m%d')
end_date = df['date'].iloc[-1].strftime('%Y%m%d')
# Structure
save_path = f"../database/{symbol}_{historical_data_interval.replace(' ', '')}_{start_date}_to_{end_date}.parquet"

# Sauvegarder le DataFrame en fichier parquet
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

In [ ]:
save_path = "../database/NDX_1min_20050411_to_20250404.parquet"
#save_path = "../database/NDX_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)

# Display the first and last rows of the DataFrame
display(retrieved_df.head(10))
display(retrieved_df.tail(10))

Instruct the notebook to draw plot graphics inline:

In [ ]:
%matplotlib inline

Plot the close data

In [ ]:
df.plot(y='close');

There is also a utility function to plot bars as a candlestick plot. It can accept either a DataFrame or a list of bars. Here it will print the last 100 bars:

In [ ]:
util.barplot(bars[-100:], title=contract.symbol);

## Historical data with realtime updates

A new feature of the API is to get live updates for historical bars. This is done by setting `endDateTime` to an empty string and the `keepUpToDate` parameter to `True`.

Let's get some bars with an keepUpToDate subscription:

In [ ]:
bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='900 S',
        barSizeSetting='10 secs',
        whatToShow='MIDPOINT',
        useRTH=True,
        formatDate=1,
        keepUpToDate=True)

Replot for every change of the last bar:

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def onBarUpdate(bars, hasNewBar):
    plt.close()
    plot = util.barplot(bars)
    clear_output(wait=True)
    display(plot)

bars.updateEvent += onBarUpdate

ib.sleep(10)
ib.cancelHistoricalData(bars)

Realtime bars
------------------

With ``reqRealTimeBars`` a subscription is started that sends a new bar every 5 seconds.

First we'll set up a event handler for bar updates:

In [ ]:
def onBarUpdate(bars, hasNewBar):
    print(bars[-1])

Then do the real request and connect the event handler,

In [ ]:
bars = ib.reqRealTimeBars(contract, 5, 'MIDPOINT', False)
bars.updateEvent += onBarUpdate

let it run for half a minute and then cancel the realtime bars.

In [ ]:
ib.sleep(30)
ib.cancelRealTimeBars(bars)

The advantage of reqRealTimeBars is that it behaves more robust when the connection to the IB server farms is interrupted. After the connection is restored, the bars from during the network outage will be backfilled and the live bars will resume.

reqHistoricalData + keepUpToDate will, at the moment of writing, leave the whole API inoperable after a network interruption.

### Request historical market news

In [ ]:
news_providers = ib.reqNewsProviders()
print("News Providers:", news_providers)
for provider in news_providers:
    print(f"Code: {provider.code}, Name: {provider.name}")

In [ ]:
article = ib.reqNewsArticle(providerCode="BRFG")
print(article)

In [ ]:
historical_news = ib.reqHistoricalNews(
    conId=contract.conId,  # ID du contrat
    providerCodes="BRFG",
    startDateTime="20250401 00:00:00",
    endDateTime="20250405 23:59:59",
    totalResults=10
)
for news in historical_news:
    print(f"Date: {news.time}, Title: {news.headline}")

In [ ]:
newsbulletin = ib.reqNewsBulletins(allMessages=True)

In [ ]:
ib.disconnect()

## Additional features

#### Data pre processing

In [1]:
import os
import pandas as pd

In [14]:
save_path = "../database/NDX_20secs_20220131_to_20250403.parquet"
df = pd.read_parquet(save_path)
display(df.head())
print(df.shape)

,date,open,high,low,close,barCount
0,2022-01-31 09:30:00-05:00,14505.07,14505.63,14504.30,14505.60,5
1,2022-01-31 09:30:20-05:00,14507.39,14514.01,14503.46,14506.64,20
2,2022-01-31 09:30:40-05:00,14508.37,14516.92,14508.37,14513.43,20
3,2022-01-31 09:31:00-05:00,14514.76,14516.97,14490.19,14490.19,20
4,2022-01-31 09:31:20-05:00,14493.16,14494.66,14478.86,14478.86,19


(929250, 6)


In [9]:
def resample_ohlc_data(input_path, output_path=None, target_interval='20S', handle_nan='drop'):
    """
    Resample OHLC data from a parquet file to a target interval.
    
    Parameters:
    -----------
    input_path : str
        Path to input parquet file
    output_path : str, optional
        Path to save output parquet file. If None, constructs a name based on input file.
    target_interval : str, optional
        Target interval for resampling (e.g., '20S', '1min', '5min')
    handle_nan : str, optional
        Method to handle NaN values: 'drop' or 'ffill' or 'interpolate'
    
    Returns:
    --------
    pd.DataFrame
        Resampled dataframe
    """
    # Read the parquet file
    df = pd.read_parquet(input_path)
    
    # Make sure the index is a datetime
    if 'date' in df.columns:
        df = df.set_index('date')
    
    # Define the resampling rules for OHLC data
    resampler = df.resample(target_interval)
    
    # Create resampled dataframe with correct OHLC aggregation
    resampled_df = pd.DataFrame({
        'open': resampler['open'].first(),
        'high': resampler['high'].max(),
        'low': resampler['low'].min(),
        'close': resampler['close'].last(),
    })
    
    # Add volume if it exists in the original dataframe
    if 'volume' in df.columns:
        resampled_df['volume'] = resampler['volume'].sum()
    
    # Add average if it exists in the original dataframe
    if 'average' in df.columns:
        resampled_df['average'] = resampler['average'].mean()
    
    # Add barCount if it exists in the original dataframe
    if 'barCount' in df.columns:
        resampled_df['barCount'] = resampler['barCount'].sum()
    
    # Handle NaN values based on the specified method
    if handle_nan == 'drop':
        resampled_df = resampled_df.dropna()
    elif handle_nan == 'ffill':
        resampled_df = resampled_df.fillna(method='ffill')
    elif handle_nan == 'interpolate':
        resampled_df = resampled_df.interpolate(method='linear')
    
    # Reset index to make date a column again
    resampled_df = resampled_df.reset_index()
    
    # Generate output filename if not provided
    if output_path is None:
        # Extract components from input path
        base_name = os.path.basename(input_path)
        dirname = os.path.dirname(input_path)
        parts = base_name.split("_")
        
        # Format the interval for the filename
        if target_interval.endswith('S') or target_interval.endswith('s'):
            # Extract the numeric part (without the 'S' or 's')
            interval_value = target_interval.rstrip('Ss')
            parts[1] = f"{interval_value}secs"
        else:
            # For other intervals (minutes, hours, etc.)
            parts[1] = target_interval.replace(' ', '')
            
        new_name = "_".join(parts)
        output_path = os.path.join(dirname, new_name)
    
    # Save the resampled data
    resampled_df.to_parquet(output_path, index=False)
    print(f"Resampled data saved to: {output_path}")
    
    return resampled_df

# Example usage:
input_file = "../database/NDX_10secs_20220131_to_20250403.parquet"

# Will create a file with 20secs interval in the name
resampled_df = resample_ohlc_data(input_file, target_interval='20s', handle_nan='drop')

# Display the first few rows of the resampled data
display(resampled_df.head())

Resampled data saved to: ../database/NDX_20secs_20220131_to_20250403.parquet


,date,open,high,low,close,barCount
0,2022-01-31 09:30:00-05:00,14505.07,14505.63,14504.30,14505.60,5
1,2022-01-31 09:30:20-05:00,14507.39,14514.01,14503.46,14506.64,20
2,2022-01-31 09:30:40-05:00,14508.37,14516.92,14508.37,14513.43,20
3,2022-01-31 09:31:00-05:00,14514.76,14516.97,14490.19,14490.19,20
4,2022-01-31 09:31:20-05:00,14493.16,14494.66,14478.86,14478.86,19
